In [363]:
import numpy as numpy
import pandas as pd 
import tensorflow as tf

In [364]:
tf.__version__

'2.21.0'

In [365]:
data = pd.read_csv("messy_ecommerce_sales_data.csv")


In [366]:
data.isnull().sum()

ID                 0
 Customer_Name     0
Order_ID           0
Order_Date         0
Product            0
 Category          8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

In [367]:
data[' Category']=data[' Category'].fillna(data[' Category'].mode()[0])

In [368]:
data['Quantity'] = pd.to_numeric(data['Quantity'],errors='coerce')
data['Quantity'] = data['Quantity'].fillna(data['Quantity'].median())

In [369]:
data['Price'] = pd.to_numeric(data['Price'],errors='coerce')
data['Price']=data['Price'].fillna(data['Price'].median())

In [370]:
data['Total'] = data['Quantity']*data['Price']

In [371]:
data = data[data['Quantity']>=0]

In [372]:
(data['Quantity']<0).sum()

np.int64(0)

In [373]:
data['Order_Date'] = pd.to_datetime(data['Order_Date'], errors='coerce')

In [374]:
data.isnull().sum()

ID                0
 Customer_Name    0
Order_ID          0
Order_Date        2
Product           0
 Category         0
Quantity          0
Price             0
Payment_Method    0
Status            0
Total             0
dtype: int64

In [375]:
data['Year'] = data['Order_Date'].dt.year
data['Month'] = data['Order_Date'].dt.month
data['Day'] = data['Order_Date'].dt.day
data['DayOfWeek'] = data['Order_Date'].dt.dayofweek

In [376]:
data['DayOfWeek']

0      4.0
1      5.0
2      0.0
3      2.0
4      0.0
      ... 
98     6.0
99     2.0
100    0.0
101    3.0
102    2.0
Name: DayOfWeek, Length: 101, dtype: float64

In [377]:
data = data.drop('Order_Date', axis=1)

In [378]:
X = data.drop(['ID', ' Customer_Name', 'Order_ID', 'Status'], axis=1)
y = data['Status']

In [379]:
print(X)

           Product     Category  Quantity    Price    Payment_Method    Total  \
0          Blender         Home       3.0   38.000  Cash on Delivery   114.00   
1       Smartphone  Electronics       2.0  542.195            PayPal  1084.39   
2    Tennis Racket       Sports       1.0  389.050            PayPal   389.05   
3          Science        Books       5.0  233.920            PayPal  1169.60   
4        Biography        Books       1.0  552.510  Cash on Delivery   552.51   
..             ...          ...       ...      ...               ...      ...   
98          Vacuum        Books       2.0  497.010  Cash on Delivery   994.02   
99         Blender         Home       5.0  372.280       Credit Card  1861.40   
100     Headphones  Electronics       1.0  111.360       Credit Card   111.36   
101          Shoes     Clothing       5.0  645.260       Credit Card  3226.30   
102     Basketball   electronic       2.0  705.420     Bank Transfer  1410.84   

       Year  Month   Day  D

In [380]:
X[' Category'] = X[' Category'].str.lower()
X['Product'] = X['Product'].str.lower()
X['Payment_Method'] = X['Payment_Method'].str.lower()

In [381]:
print(X)

           Product     Category  Quantity    Price    Payment_Method    Total  \
0          blender         home       3.0   38.000  cash on delivery   114.00   
1       smartphone  electronics       2.0  542.195            paypal  1084.39   
2    tennis racket       sports       1.0  389.050            paypal   389.05   
3          science        books       5.0  233.920            paypal  1169.60   
4        biography        books       1.0  552.510  cash on delivery   552.51   
..             ...          ...       ...      ...               ...      ...   
98          vacuum        books       2.0  497.010  cash on delivery   994.02   
99         blender         home       5.0  372.280       credit card  1861.40   
100     headphones  electronics       1.0  111.360       credit card   111.36   
101          shoes     clothing       5.0  645.260       credit card  3226.30   
102     basketball   electronic       2.0  705.420     bank transfer  1410.84   

       Year  Month   Day  D

In [382]:
X[' Category'] = X[' Category'].replace({
    'electronic':'electronics'
})

In [383]:
X = X.drop('Total', axis=1)

In [384]:
X.columns = X.columns.str.strip()

In [385]:
X = pd.get_dummies(X, columns=['Product', 'Category', 'Payment_Method'], drop_first=True)

In [386]:
print(X.columns)

Index(['Quantity', 'Price', 'Year', 'Month', 'Day', 'DayOfWeek',
       'Product_biography', 'Product_blender', 'Product_comics',
       'Product_fiction', 'Product_football', 'Product_headphones',
       'Product_jacket', 'Product_jeans', 'Product_lamp', 'Product_laptop',
       'Product_microwave', 'Product_science', 'Product_shoes',
       'Product_smartphone', 'Product_smartwatch', 'Product_t-shirt',
       'Product_tennis racket', 'Product_vacuum', 'Product_yoga mat',
       'Category_clothing', 'Category_electronics', 'Category_home',
       'Category_sports', 'Payment_Method_cash on delivery',
       'Payment_Method_credit card', 'Payment_Method_paypal'],
      dtype='object')


In [387]:
X['Year'] = X['Year'].fillna(X['Year'].median())
X['Month'] = X['Month'].fillna(X['Month'].median())
X['Day'] = X['Day'].fillna(X['Day'].median())
X['DayOfWeek'] = X['DayOfWeek'].fillna(X['DayOfWeek'].median())

In [388]:
X = X.astype(int)

In [389]:
print(X)

     Quantity  Price  Year  Month  Day  DayOfWeek  Product_biography  \
0           3     38  2024     11   22          4                  0   
1           2    542  2025      7    5          5                  0   
2           1    389  2024     12   23          0                  0   
3           5    233  2025      3   19          2                  0   
4           1    552  2025     10   20          0                  1   
..        ...    ...   ...    ...  ...        ...                ...   
98          2    497  2025      7   27          6                  0   
99          5    372  2025      1   22          2                  0   
100         1    111  2025      2   24          0                  0   
101         5    645  2025     10   30          3                  0   
102         2    705  2025      7    9          2                  0   

     Product_blender  Product_comics  Product_fiction  ...  \
0                  1               0                0  ...   
1          

In [390]:
cols_to_drop = [col for col in X.columns if 'Product_' in col]
X = X.drop(cols_to_drop, axis=1)

In [416]:
def simplify_status(x):
    if x in ['delivered', 'shipped']:
        return 'Success'
    elif x in ['cancelled', 'returned']:
        return 'Failure'
    else:
        return 'Processing'

In [417]:
y = data['Status'].str.lower().apply(simplify_status)

In [392]:
X['Price_per_item'] = X['Price'] / X['Quantity']

In [393]:
X['is_weekend'] = X['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

In [420]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [427]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [455]:
from sklearn.svm import SVC

model = SVC(kernel='linear')   
model.fit(X_train, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [456]:
y_pred = model.predict(X_test)

In [457]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print("Accuracy : ",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

Accuracy :  0.47619047619047616
              precision    recall  f1-score   support

     Failure       0.60      0.67      0.63         9
  Processing       0.20      0.20      0.20         5
     Success       0.50      0.43      0.46         7

    accuracy                           0.48        21
   macro avg       0.43      0.43      0.43        21
weighted avg       0.47      0.48      0.47        21

[[6 2 1]
 [2 1 2]
 [2 2 3]]


In [432]:
y.value_counts()

Status
Failure       42
Success       33
Processing    26
Name: count, dtype: int64